In [4]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
client = Anthropic()



In [5]:
load_dotenv(override=True)     # fuerza la sobrescritura

True

In [6]:
import os
from dotenv import dotenv_values

archivo = dotenv_values(".env").get("ANTHROPIC_API_KEY")
entorno = os.getenv("ANTHROPIC_API_KEY")

print("archivo:", repr(archivo[:14]), len(archivo))
print("entorno:", repr(entorno[:14]), len(entorno))
print("iguales:", archivo == entorno)

archivo: 'sk-ant-api03-V' 108
entorno: 'sk-ant-api03-V' 108
iguales: True


In [7]:
def llm(prompt):
    response = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

In [8]:
#llm("Hey, what's up?")

Next session uv sync                  # reconstruye el entorno desde uv.lock
Hay que reactivar la api 



 # RAG Agent architecture

In [9]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [10]:
#print(prompt)

In [11]:
#question = "I just discovered the course. Can I join now?"
#answer = llm(prompt)
#print(answer)

In [12]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)    
    return llm(user_prompt)

In [13]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [14]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1380

documents[0]

In [15]:
#minisearch is based on elasticsearch, but it is a lightweight version that can be used in memory without the need for a separate server. 
#It allows you to create an index of documents and perform searches on them.
#We can see the structure of the documents and the fields that we can use for indexing and searching in the cell above. 
# The documents have the following fields: "question", "section", "answer", and "course". We can use these fields to create an index and perform searches on them.
#Keyword fields are used for exact matches, while text fields are used for full-text search. In this case, we can use the "course" field as a keyword field and the "question", "section", and "answer" fields as text fields.
#keyword can help you to filter the docs by course, restricting the search to a specific course. This can be useful if you want to find answers to questions related to a specific course, rather than searching through all the courses.

from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [16]:
question = "I just discovered the course. Can I join now?"

#search_results = index.search(
#    question,
#    boost_dict={"question": 2.0, "section": 0.5},
#    filter_dict={"course": "llm-zoomcamp"},
#    num_results=5
#)

#search_results

In [17]:
#search_results = index.search(question, filter_dict={"course": "llm-zoomcamp"}, num_results=5)

In [18]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )
    
search_results = search(question)

## Pompt


In [19]:
#Divide pormpt into two: the instructions and the user prompt that changes with every request

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [20]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [21]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [22]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [23]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chaining tasks together. That's what an orchest

## RAG pipeline


This is to se the tokens used 
reponse.usage 

In [24]:
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [25]:
response.content[0].text
print(response)

Message(id='msg_011CdUhayHTCyfpJ7fgf6NvG', container=None, content=[TextBlock(citations=None, text="# Yes, you can join now!\n\nBased on the course guidelines:\n\n✅ **You can start learning immediately** — all videos and materials are available, and you can begin working through the content at your own pace.\n\n✅ **You can submit homework** — while the submission form is open, you can complete and submit assignments.\n\n⚠️ **Certificate requirement** — If you want to earn a certificate, you need to:\n- Submit your capstone project **while a live cohort is accepting submissions**\n- Complete the required peer reviews during that same active period\n\n**Next steps:**\n1. Check the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp) to get started\n2. Review the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for current deadlines\n3. Follow the typical workflow: watch videos → work through code → complete homework → submit befor

In [26]:
response.usage

Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=760, output_tokens=245, output_tokens_details=None, server_tool_use=None, service_tier='standard')

In [27]:
input_price = 1.00 / 1_000_000
output_price = 5.00 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0019850000000000002

In [28]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS}, #system prompt constant, in "developer" you can change it to "system" 
    {"role": "user", "content": prompt} #user prompt variable
]

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [29]:
def llm(instructions, user_prompt, model="claude-haiku-4-5"):
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        system=instructions,
        messages=[
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.content[0].text

In [30]:
def rag(query, model="claude-haiku-4-5"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [31]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

# Yes, you can join now!

According to the course information:

**You can start learning immediately.** The videos and GitHub materials are available, and you can begin working through the content whenever you want.

**However, regarding certificates:** If you want to receive a certificate, you'll need to:
- Submit your capstone project while the course is still accepting submissions
- Complete the required peer reviews

You can work through the material and prepare your project in self-paced mode, but **project submission and peer review must happen while a live cohort is actively accepting them** (certificates are only awarded for completing the course with a "live" cohort, not in fully self-paced mode).

To get started, check out:
- [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)
- [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)
- [Course deadlines](https://courses.datatalks.club/llm-zoomcamp-2026/)

The typical workflow is: wat

In [32]:
answer = rag("What can you tell me about the course?")
print(answer)

# About the Course

Based on the available information, here's what I can tell you about the course:

## Course Structure
- This is the **LLM Zoomcamp** - a structured learning program
- The course follows a **weekly workflow** with video lessons, hands-on notebooks/code, and homework assignments
- You can start whenever you want, as videos and materials are available on GitHub

## Certificate Requirements
- Certificates are **only available** if you complete the course with a "live" cohort
- To earn a certificate, you must:
  - Complete a **capstone project** and submit it before the submission deadline
  - Complete **required peer reviews**
  - Note: Homework is optional for certification

## Getting Started
To begin, you should:
1. Check the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)
2. Review the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/)
3. Access the [LLM Zoomcamp GitHub repository](https://github.com/Dat